In [7]:
!pip install simpy
import random

WAIT_TIMES = []

def customer(env, name, server, service_time):
    arrival_time = env.now
    print(f"{name} arrives at {arrival_time:.2f}")

    with server.request() as request:
        yield request
        wait = env.now - arrival_time
        WAIT_TIMES.append(wait)
        print(f"{name} starts service at {env.now:.2f} (Waited: {wait:.2f})")
        yield env.timeout(random.randint(1, service_time))
        print(f"{name} leaves at {env.now:.2f}")

def customer_generator(env, server, inter_arrival, service_time):
    i = 0
    while True:
        yield env.timeout(random.randint(1, inter_arrival))
        i += 1
        env.process(customer(env, f"Customer {i}", server, service_time))

# Setup
env= simpy.Environment()
server = simpy.Resource(env, capacity=1) # Single server
env.process(customer_generator(env, server, inter_arrival=5, service_time=3))
env.run(until=20)

Customer 1 arrives at 3.00
Customer 1 starts service at 3.00 (Waited: 0.00)
Customer 1 leaves at 6.00
Customer 2 arrives at 8.00
Customer 2 starts service at 8.00 (Waited: 0.00)
Customer 3 arrives at 10.00
Customer 2 leaves at 10.00
Customer 3 starts service at 10.00 (Waited: 0.00)
Customer 4 arrives at 13.00
Customer 3 leaves at 13.00
Customer 4 starts service at 13.00 (Waited: 0.00)
Customer 5 arrives at 14.00
Customer 4 leaves at 14.00
Customer 5 starts service at 14.00 (Waited: 0.00)
Customer 5 leaves at 15.00
Customer 6 arrives at 19.00
Customer 6 starts service at 19.00 (Waited: 0.00)


In [4]:
import simpy
import random

def customer(env, name, server, service_time):
    arrival_time = env.now
    with server.request() as request:
        yield request
        print(f"{name} served by one of the 2 servers at {env.now:.2f}")
        yield env.timeout(random.randint(1, service_time))

def generator(env, server):
    for i in range(5):
        yield env.timeout(random.randint(1, 3))
        env.process(customer(env, f"Cust {i}", server, 5))

env = simpy.Environment()
# Change capacity to 2 for a two-server system
server = simpy.Resource(env, capacity=2)
env.process(generator(env, server))
env.run()

Cust 0 served by one of the 2 servers at 3.00
Cust 1 served by one of the 2 servers at 5.00
Cust 2 served by one of the 2 servers at 7.00
Cust 3 served by one of the 2 servers at 10.00
Cust 4 served by one of the 2 servers at 11.00


In [5]:
import simpy
import random

# Example of high traffic (Short inter-arrival, long service time)
INTER_ARRIVAL = 2
SERVICE_TIME = 10

def customer(env, name, server):
    with server.request() as request:
        yield request
        yield env.timeout(random.randint(1, SERVICE_TIME))

def generator(env, server):
    for i in range(5):
        yield env.timeout(random.randint(1, INTER_ARRIVAL))
        env.process(customer(env, f"C{i}", server))

env = simpy.Environment()
res = simpy.Resource(env, capacity=1)
env.process(generator(env, res))
env.run()
print("Simulation finished with high-traffic parameters.")

Simulation finished with high-traffic parameters.


In [6]:
import simpy
import random

WAIT_TIMES = []

def customer(env, name, server):
    arrival = env.now
    with server.request() as request:
        yield request
        WAIT_TIMES.append(env.now - arrival)
        yield env.timeout(random.randint(1, 5))

def generator(env, server):
    for i in range(10):
        yield env.timeout(random.randint(1, 3))
        env.process(customer(env, f"C{i}", server))

env = simpy.Environment()
server = simpy.Resource(env, capacity=1)
env.process(generator(env, server))
env.run()

# Calculate and display Average
if WAIT_TIMES:
    avg_wait = sum(WAIT_TIMES) / len(WAIT_TIMES)
    print(f"\nTotal Customers: {len(WAIT_TIMES)}")
    print(f"Average Waiting Time: {avg_wait:.2f} units")


Total Customers: 10
Average Waiting Time: 4.50 units
